In [1]:
import os
import json
import time
import zipfile
import datetime
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
from utilis import Utility, Data, Visualization, Score

/Users/viethuy/Working_space/Neurips/Neurips2025_Weak_lensing/utilis.py:408: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  DEVICE = "mps" if torch.has_mps else "cpu"


In [3]:
root_dir = os.getcwd()
print("Root directory is", root_dir)

Root directory is /Users/viethuy/Working_space/Neurips/Neurips2025_Weak_lensing


In [4]:
USE_PUBLIC_DATASET = True
DATA_DIR ='./dataset'  # This is only required when you set USE_PUBLIC_DATASET = True

In [5]:
# Initialize Data class object
data_obj = Data(data_dir=DATA_DIR, USE_PUBLIC_DATASET=USE_PUBLIC_DATASET)
data_obj.load_train_data()
# Load test data
data_obj.load_test_data()

Loading WIDE12H_bin2_2arcmin_kappa.npy:   0%|          | 0/11 [00:00<?, ?it/s]

Loading WIDE12H_bin2_2arcmin_kappa.npy: 100%|██████████| 11/11 [00:54<00:00,  4.94s/it]
Loading WIDE12H_bin2_2arcmin_kappa_noisy_test.npy: 100%|██████████| 40/40 [00:06<00:00,  6.17it/s]


In [6]:
Ncosmo = data_obj.Ncosmo
Nsys = data_obj.Nsys

print(f'There are {Ncosmo} cosmological models, each has {Nsys} realizations of nuisance parameters in the training data.')
print(f'Shape of the training data = {data_obj.kappa.shape}')
print(f'Shape of the mask = {data_obj.mask.shape}')
print(f'Shape of the training label = {data_obj.label.shape}')
print(f'Shape of the test data = {data_obj.kappa_test.shape}')

There are 101 cosmological models, each has 256 realizations of nuisance parameters in the training data.
Shape of the training data = (101, 256, 1424, 176)
Shape of the mask = (1424, 176)
Shape of the training label = (101, 256, 5)
Shape of the test data = (4000, 1424, 176)


In [7]:
import os
import numpy as np

# --- Cấu hình ---
num_chunks = 25
seed = 113  # để tái lập kết quả
save_dir = './dataset/chunk_kappa_noise_batchnorm'  # nơi lưu các chunk noisy + label
os.makedirs(save_dir, exist_ok=True)

# --- Kiểm tra dữ liệu nguồn ---
assert hasattr(data_obj, 'kappa') and hasattr(data_obj, 'label'), 'Cần load data_obj trước (data_obj.kappa, data_obj.label)'
Ncosmo, Nsys = data_obj.kappa.shape[0], data_obj.kappa.shape[1]
print(f'Ncosmo={Ncosmo}, Nsys={Nsys}')

# Tính kích thước cơ bản và phần dư để phân phối đều
base_size = Nsys // num_chunks  # thường = 10 cho Nsys=256, num_chunks=25
remainder = Nsys % num_chunks   # phần dư (ví dụ 6)
print(f'base_size={base_size}, remainder={remainder} (tức sẽ có {remainder} chunk có base_size+1 samples)')

# Tạo permutation reproducible (sử dụng cùng một permutation cho mọi cosmology)
rng = np.random.default_rng(seed)
perm = rng.permutation(Nsys)

# Lưu chunk: mỗi chunk có shape (Ncosmo, chunk_size, H, W) và label (Ncosmo, chunk_size, K)
pos = 0
for i in range(num_chunks):
    size = base_size + (1 if i < remainder else 0)
    idx = perm[pos: pos + size]
    pos += size

    noisy_chunk = data_obj.kappa[:, idx]  # (Ncosmo, size, H, W)
    label_chunk = data_obj.label[:, idx]  # (Ncosmo, size, K)

    noisy_path = os.path.join(save_dir, f'kappa_noisy_chunk_{i}.npy')
    label_path = os.path.join(save_dir, f'label_chunk_{i}.npy')
    np.save(noisy_path, noisy_chunk)
    np.save(label_path, label_chunk)
    print(f'Saved chunk {i}: noisy {noisy_chunk.shape} -> {noisy_path}; label {label_chunk.shape} -> {label_path}')

if pos != Nsys:
    print(f'Warning: used {pos} indices out of {Nsys} (leftover {Nsys-pos}).')
else:
    print('All indices distributed across chunks.')

# Lưu ý:
# - Với Nsys=256, num_chunks=25: base_size=10, remainder=6 -> 6 chunk đầu có 11 samples/cosmo, 19 chunk sau có 10.
# - Mỗi chunk vẫn chứa đủ tất cả cosmologies (Ncosmo) nên bạn có thể nạp từng chunk và huấn luyện lần lượt.
# - Nếu bạn muốn bỏ phần ngẫu nhiên và dùng slicing tuần tự (0..9, 10..19, ...), thay perm bằng np.arange(Nsys).
# - Nếu bạn muốn chunk có kích thước cố định 10 và bỏ phần dư, đặt remainder=0 hoặc chỉ lấy perm[:num_chunks*10].
# - Sau khi tạo chunk bạn có thể dùng load_single_chunk(chunk_idx, save_dir) hoặc np.load để nạp từng chunk vào huấn luyện.

Ncosmo=101, Nsys=256
base_size=10, remainder=6 (tức sẽ có 6 chunk có base_size+1 samples)
Saved chunk 0: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_batchnorm/kappa_noisy_chunk_0.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_batchnorm/label_chunk_0.npy
Saved chunk 1: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_batchnorm/kappa_noisy_chunk_1.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_batchnorm/label_chunk_1.npy
Saved chunk 2: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_batchnorm/kappa_noisy_chunk_2.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_batchnorm/label_chunk_2.npy
Saved chunk 3: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_batchnorm/kappa_noisy_chunk_3.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_batchnorm/label_chunk_3.npy
Saved chunk 4: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_batchnorm/kappa_noisy_chunk_4.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_batchnor

# seperate the dataset to many chunk with norm each chunk


In [8]:
import os
import numpy as np

# Cấu hình
num_chunks = 25
seed = 113
save_dir = './dataset/chunk_kappa_noise_new'  # nơi lưu noisy chunks và label chunks
os.makedirs(save_dir, exist_ok=True)

# Kiểm tra data_obj và Utility
assert 'data_obj' in globals(), 'Chưa load data_obj; hãy chạy cell tạo Data object trước.'
assert 'Utility' in globals(), 'Chưa import Utility; hãy chạy from utilis import Utility, Data, ...'

Ncosmo, Nsys = data_obj.kappa.shape[0], data_obj.kappa.shape[1]
print(f'Ncosmo={Ncosmo}, Nsys={Nsys}')

# Tạo permutation reproducible (dùng cùng một permutation cho mọi cosmology để đảm bảo các chunk có đủ cosmo)
rng = np.random.default_rng(seed)
perm = rng.permutation(Nsys)

# Kích thước chunk
base_size = Nsys // num_chunks
remainder = Nsys % num_chunks
print(f'base_size={base_size}, remainder={remainder}')

# Lặp và xử lý từng chunk
pos = 0
for i in range(num_chunks):
    size = base_size + (1 if i < remainder else 0)
    if size == 0:
        print(f'Chunk {i} has size 0, skipping')
        continue
    idx = perm[pos: pos + size]
    pos += size

    # Lấy phần tử của kappa và label tương ứng (shape: (Ncosmo, size, H, W), (Ncosmo, size, K))
    kappa_chunk = data_obj.kappa[:, idx]
    label_chunk = data_obj.label[:, idx]

    # Gọi hàm add_noise (hàm này đã xuất hiện trong Utility của bạn)
    noisy_chunk = Utility.add_noise(
        data=kappa_chunk.astype(np.float64),
        mask=data_obj.mask,
        ng=data_obj.ng,
        pixel_size=data_obj.pixelsize_arcmin
    )

    # Lưu noisy và label chunk
    noisy_path = os.path.join(save_dir, f'kappa_noisy_chunk_{i}.npy')
    label_path = os.path.join(save_dir, f'label_chunk_{i}.npy')
    np.save(noisy_path, noisy_chunk)
    np.save(label_path, label_chunk)

    print(f'Saved chunk {i}: noisy {noisy_chunk.shape} -> {noisy_path}; label {label_chunk.shape} -> {label_path}')

if pos != Nsys:
    print(f'Warning: used {pos} indices out of {Nsys} (leftover {Nsys-pos}).')
else:
    print('All indices distributed across chunks.')

# Khi muốn nạp từng chunk trong quá trình huấn luyện dùng load_single_chunk hoặc np.load:
# noisy_chunk, label_chunk = load_single_chunk(0, save_dir) # hoặc np.load(...)

Ncosmo=101, Nsys=256
base_size=10, remainder=6
Saved chunk 0: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_noisy_chunk_0.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_new/label_chunk_0.npy
Saved chunk 1: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_noisy_chunk_1.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_new/label_chunk_1.npy
Saved chunk 2: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_noisy_chunk_2.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_new/label_chunk_2.npy
Saved chunk 3: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_noisy_chunk_3.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_new/label_chunk_3.npy
Saved chunk 4: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_noisy_chunk_4.npy; label (101, 11, 5) -> ./dataset/chunk_kappa_noise_new/label_chunk_4.npy
Saved chunk 5: noisy (101, 11, 1424, 176) -> ./dataset/chunk_kappa_noise_new/kappa_

In [7]:
# # Directory to save chunks
# !mkdir -p ./dataset/chunk_kappa
# save_dir = './dataset/chunk_kappa'
# os.makedirs(save_dir, exist_ok=True)

# # Number of chunks you want
# num_chunks = 10  # Change this as needed

# # Split kappa along the first axis (Ncosmo)
# chunks = np.array_split(data_obj.kappa, num_chunks, axis=0)

# for idx, chunk in enumerate(chunks):
#     chunk_path = os.path.join(save_dir, f'kappa_chunk_{idx}.npy')
#     np.save(chunk_path, chunk)
#     print(f'Saved {chunk_path}, shape: {chunk.shape}')

In [ ]:
# Directory to save noisy chunks
!mkdir -p ./dataset/chunk_kappa_noise
save_dir = './dataset/chunk_kappa_noise'
os.makedirs(save_dir, exist_ok=True)

# Number of chunks
num_chunks = 10  # Adjust as needed

np.random.seed(113)  # For reproducibility

chunks = np.array_split(data_obj.kappa, num_chunks, axis=0)

for idx, chunk in enumerate(chunks):
    noisy_chunk = Utility.add_noise(
        data=chunk.astype(np.float64),
        mask=data_obj.mask,
        ng=data_obj.ng,
        pixel_size=data_obj.pixelsize_arcmin
    )
    chunk_path = os.path.join(save_dir, f'kappa_noisy_chunk_{idx}.npy')
    np.save(chunk_path, noisy_chunk)
    print(f'Saved {chunk_path}, shape: {noisy_chunk.shape}')

In [ ]:
import numpy as np
import os

# Đường dẫn tới thư mục chứa các chunk noisy và file label gốc
chunk_dir = './chunk_kappa_noise'
label_path = './label.npy'

# Lấy danh sách các file chunk noisy, đảm bảo đúng thứ tự
chunk_files = sorted([f for f in os.listdir(chunk_dir) if f.startswith('kappa_noisy_chunk_') and f.endswith('.npy')])

# Nạp toàn bộ label
labels = np.load(label_path)

start = 0
for chunk_file in chunk_files:
    chunk_path = os.path.join(chunk_dir, chunk_file)
    noisy_chunk = np.load(chunk_path)
    num_samples = noisy_chunk.shape[0]
    labels_chunk = labels[start:start+num_samples]
    
    # Lưu file nhãn tương ứng
    label_chunk_name = chunk_file.replace('kappa_noisy_chunk_', 'label_chunk_')
    label_chunk_path = os.path.join(chunk_dir, label_chunk_name)
    np.save(label_chunk_path, labels_chunk)
    
    print(f"Saved {label_chunk_path} with shape {labels_chunk.shape}")
    start += num_samples